# 📐 SAiDL 2026 — Positional Extrapolation Test & Convolution Hybrids

This notebook is **exclusively** for two tasks from the SAiDL Core ML assignment:

| Section | Task | Assignment Ref |
|---------|------|---------------|
| **A** | Positional Extrapolation Test (Phase 1.3) | Train on L=512 → eval at L∈{512,1024,2048} for every PE scheme |
| **B** | Convolution + Attention Hybrids (Phase 1.4) | All 3 hybrid types vs pure-attention best checkpoint |

**Pre-requisite:** Run `SAiDL_colab.ipynb` Sections 0 and 1 first so that:
- The repo is cloned at `/content/SAiDL-Summer-Assignment-2026`
- All packages are installed
- W&B is authenticated

All cells print a clear ✅/❌ and full tracebacks on failure.

---
## 0 — Environment Bootstrap

In [ ]:
# ── 0A  Anchor repo path ──────────────────────────────────────────────────────
import os, sys, traceback

REPO = '/content/SAiDL-Summer-Assignment-2026'

try:
    assert os.path.exists(REPO), (
        f'Repo not found at {REPO}. '
        'Run SAiDL_colab.ipynb Section 0 first.'
    )
    os.chdir(REPO)
    sys.path.insert(0, REPO)
    print(f'✅ Repo found at {REPO}')
except Exception as e:
    print(f'❌ FAILED: {e}')
    traceback.print_exc()

: 

In [ ]:
# ── 0B  Install / verify packages ────────────────────────────────────────────
import subprocess

PACKAGES = [
    'transformers', 'datasets', 'wandb', 'hydra-core',
    'omegaconf', 'einops', 'tqdm', 'matplotlib', 'numpy'
]

try:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + PACKAGES,
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('❌ pip install failed:')
        print(result.stderr)
    else:
        print('✅ All packages verified')
except Exception as e:
    print(f'❌ FAILED: {e}')
    traceback.print_exc()

In [ ]:
# ── 0C  W&B login ────────────────────────────────────────────────────────────
try:
    import wandb
    wandb.login()  # paste key from https://wandb.ai/authorize
    print('✅ W&B logged in')
except Exception as e:
    print(f'❌ FAILED: {e}')
    traceback.print_exc()

In [ ]:
# ── 0D  Verify all critical imports ──────────────────────────────────────────
checks = [
    ('core_ml.data.dataset',                      'LanguageModelingDataset'),
    ('core_ml.models.transformer',                'Transformer'),
    ('core_ml.models.blocks',                     'TransformerBlock'),
    ('core_ml.models.ffn',                        'FeedForward'),
    ('core_ml.models.attention.vanilla_attention','MultiHeadAttention'),
    ('core_ml.models.attention.sliding_window',   'SlidingWindowAttention'),
    ('core_ml.models.attention.gqa',              'GroupedQueryAttention'),
    ('core_ml.models.attention.relu_attention',   'ReLUAttention'),
    ('core_ml.models.attention.Sparse_attention', 'SparseAttention'),
    ('core_ml.models.positional.Sinusoidal',      'SinusoidalPositionalEncoding'),
    ('core_ml.models.positional.Rope',            'RotaryPositionalEmbedding'),
    ('core_ml.models.positional.Alibi',           'ALiBiPositionalBias'),
    ('core_ml.models.positional.RelativeBias',    'RelativePositionalBias'),
    ('core_ml.models.hybrid.hybrid_blocks',       'ConvBeforeAttnBlock'),
    ('core_ml.models.hybrid.hybrid_blocks',       'GatedConvFFNBlock'),
    ('core_ml.models.hybrid.hybrid_blocks',       'PureConvBlock'),
    ('core_ml.train.train',                       'build_model'),
    ('core_ml.train.trainer',                     'Trainer'),
    ('core_ml.train.dataset',                     'prepare_dataloaders'),
]

all_ok = True
for module_path, symbol in checks:
    try:
        mod = __import__(module_path, fromlist=[symbol])
        getattr(mod, symbol)
        print(f'  ✅ {module_path}.{symbol}')
    except Exception as e:
        print(f'  ❌ {module_path}.{symbol}  →  {e}')
        all_ok = False

print()
if all_ok:
    print('✅ All imports OK — safe to run experiments')
else:
    print('❌ Fix imports above before proceeding')

In [ ]:
# ── 0E  Shared experiment runner ─────────────────────────────────────────────

def run_experiment(name, overrides, script='core_ml/train/train.py'):
    """
    Generic subprocess launcher — streams stdout live and prints full
    traceback on non-zero exit.  Returns True on success.
    """
    cmd = [sys.executable, script] + overrides
    sep = '─' * 62
    print(f'
{sep}')
    print(f'  ▶  {name}')
    print(f'  CMD: {
.join(cmd)}')
    print(f'{sep}
')

    proc = subprocess.Popen(
        cmd, cwd=REPO,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()

    if proc.returncode != 0:
        print(f'
❌  FAILED: {name}  (exit {proc.returncode})')
        print('    Scroll up for the Python traceback.')
        return False
    print(f'
✅  DONE: {name}')
    return True

print('✅ run_experiment helper ready')

---
## SECTION A — Positional Extrapolation Test (Phase 1.3)

> **Protocol (from assignment):** Train each model on `L_train = 512`.  
> Evaluate validation perplexity at `L_test ∈ {512, 1024, 2048}`  
> to measure how well each positional encoding **extrapolates** beyond its training length.

We test all four PE schemes with `vanilla_mha` attention (isolates the PE variable):

| Run | Positional Encoding | Expected extrapolation |
|-----|--------------------|-----------------------|
| A.1 | Sinusoidal | 🔴 Fails catastrophically at L > 512 |
| A.2 | RoPE | 🟡 Graceful decay, partial extrapolation |
| A.3 | ALiBi | 🟢 Best — linear bias generalises cleanly |
| A.4 | Relative Bias | 🟡 Clips at max_relative_distance=128 |

### A.1 — Train: Sinusoidal (L_train = 512)

In [ ]:
run_experiment('extrap_train__sinusoidal_L512', [
    'attention=vanilla',
    'positional=sinusoidal',
    'dataset.seq_len=512',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'training.num_epochs=10',        # short train for extrapolation study
    'model.max_seq_len=512',
    'experiment_name=extrap_sin_train512',
    'output_dir=outputs/extrap_sin',
])

### A.2 — Train: RoPE (L_train = 512)

In [ ]:
run_experiment('extrap_train__rope_L512', [
    'attention=vanilla',
    'positional=rope',
    'dataset.seq_len=512',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'training.num_epochs=10',
    'model.max_seq_len=512',
    'experiment_name=extrap_rope_train512',
    'output_dir=outputs/extrap_rope',
])

### A.3 — Train: ALiBi (L_train = 512)

In [ ]:
run_experiment('extrap_train__alibi_L512', [
    'attention=vanilla',
    'positional=alibi',
    'dataset.seq_len=512',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'training.num_epochs=10',
    'model.max_seq_len=512',
    'experiment_name=extrap_alibi_train512',
    'output_dir=outputs/extrap_alibi',
])

### A.4 — Train: Relative Bias (L_train = 512)

In [ ]:
run_experiment('extrap_train__relative_L512', [
    'attention=vanilla',
    'positional=relative',
    'dataset.seq_len=512',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'training.num_epochs=10',
    'model.max_seq_len=512',
    'experiment_name=extrap_rel_train512',
    'output_dir=outputs/extrap_rel',
])

---
### A.5 — Evaluate all four checkpoints across L_test ∈ {512, 1024, 2048}

This cell **does not re-train** — it loads the saved checkpoints and only runs inference over dataloaders built at each test sequence length.

In [ ]:
# ── Extrapolation evaluation harness ─────────────────────────────────────────
import torch
import torch.nn as nn
import math
import json
import os

from omegaconf import OmegaConf
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from datasets import load_dataset
from tqdm.auto import tqdm

from core_ml.data.dataset  import LanguageModelingDataset
from core_ml.train.train   import build_model


def build_val_loader(seq_len: int, batch_size: int = 4, num_workers: int = 2) -> DataLoader:
    """Returns a WikiText-2 validation DataLoader for a given seq_len."""
    raw       = load_dataset('wikitext', 'wikitext-2-raw-v1')
    tokenizer = AutoTokenizer.from_pretrained('distilgpt2')
    tokenized = raw.map(
        lambda ex: tokenizer(ex['text'], truncation=False, return_attention_mask=False),
        batched=True, remove_columns=['text']
    )
    val_stream = [t for sub in tokenized['validation']['input_ids'] for t in sub]
    val_ds = LanguageModelingDataset(val_stream, seq_len=seq_len)
    return DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                      num_workers=num_workers, pin_memory=True, drop_last=True)


@torch.no_grad()
def eval_perplexity(model, loader, device, use_fp16=True):
    """Returns (perplexity, tokens_per_sec)."""
    criterion  = nn.CrossEntropyLoss()
    model.eval()
    total_loss, total_tokens = 0.0, 0
    import time
    t0 = time.perf_counter()
    for x, y in tqdm(loader, desc='  eval', leave=False):
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast('cuda', enabled=(use_fp16 and device.type == 'cuda')):
            logits = model(x)
            loss   = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        total_loss   += loss.item() * x.numel()
        total_tokens += x.numel()
    elapsed = time.perf_counter() - t0
    ppl = math.exp(min(total_loss / total_tokens, 20))
    return ppl, total_tokens / elapsed


def build_cfg_for(pos_name: str, seq_len: int = 512):
    """Build a minimal OmegaConf config matching the model used during training."""
    cfg = OmegaConf.create({
        'model': {
            'name': 'baseline_transformer',
            'd_model': 512, 'n_layers': 6, 'n_heads': 8,
            'd_ff': 2048, 'dropout': 0.1,
            'vocab_size': 50257, 'max_seq_len': seq_len,
        },
        'attention': {
            'name': 'vanilla_mha', 'num_heads': 8,
            'dropout': 0.1, 'is_causal': True,
        },
        'positional': {
            'name': pos_name,
            'max_len': seq_len,       # used by sinusoidal & rope
            'base': 10000.0,
            'max_relative_distance': 128,  # used by relative
        },
    })
    return cfg


# ── Configs: (display_name, pos_name, checkpoint_dir) ────────────────────────
EXTRAP_RUNS = [
    ('Sinusoidal', 'sinusoidal', 'outputs/extrap_sin/best_model.pt'),
    ('RoPE',       'rope',       'outputs/extrap_rope/best_model.pt'),
    ('ALiBi',      'alibi',      'outputs/extrap_alibi/best_model.pt'),
    ('Relative',   'relative',   'outputs/extrap_rel/best_model.pt'),
]

TEST_LENGTHS = [512, 1024, 2048]
device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
results      = {}   # { pe_name: { seq_len: ppl } }

print(f'Device: {device}')
print(f'Evaluating PEs: {[r[0] for r in EXTRAP_RUNS]}')
print(f'Test lengths:   {TEST_LENGTHS}
')

for display_name, pos_name, ckpt_path in EXTRAP_RUNS:
    print(f'
{'═'*55}')
    print(f'  PE: {display_name}')
    print(f'{'═'*55}')
    results[display_name] = {}

    # Build model at TRAINING length (512) to load checkpoint weights
    cfg   = build_cfg_for(pos_name, seq_len=512)
    model = build_model(cfg).to(device)

    ckpt_full = os.path.join(REPO, ckpt_path)
    if os.path.exists(ckpt_full):
        state = torch.load(ckpt_full, map_location=device)
        model.load_state_dict(state)
        print(f'  Loaded checkpoint: {ckpt_path}')
    else:
        print(f'  ⚠ Checkpoint not found at {ckpt_full}. Using random init.')

    for test_len in TEST_LENGTHS:
        try:
            loader = build_val_loader(test_len, batch_size=4)
            ppl, tok_s = eval_perplexity(model, loader, device)
            results[display_name][test_len] = round(ppl, 3)
            tag = '  (in-dist)' if test_len == 512 else '  (OOD)'
            print(f'  L_test={test_len:4d}{tag}  →  PPL = {ppl:.2f}   tok/s={tok_s:.0f}')
        except Exception as e:
            print(f'  L_test={test_len} FAILED: {e}')
            results[display_name][test_len] = None

print('
✅ Extrapolation evaluation complete')

### A.6 — Extrapolation Results Table + Plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# ── 1. Print ASCII summary table ─────────────────────────────────────────────
header = f"{'PE':15s}" + ''.join(f"  L={L:<6d}" for L in TEST_LENGTHS)
print('\n' + '─'*len(header))
print('  EXTRAPOLATION RESULTS (Validation Perplexity ↓)')
print('─'*len(header))
print('  ' + header)
print('  ' + '─'*(len(header)-2))
for pe, lens in results.items():
    row = f"  {pe:15s}"
    for L in TEST_LENGTHS:
        val = lens.get(L)
        row += f"  {str(val) if val else 'N/A':<8s}"
    print(row)
print('─'*len(header) + '\n')

# ── 2. Delta-PPL table (degradation vs L=512 baseline) ───────────────────────
print('  ΔPPL vs in-distribution (L=512) — lower = better extrapolation')
print('  ' + '─'*(len(header)-2))
for pe, lens in results.items():
    base = lens.get(512)
    row  = f"  {pe:15s}"
    for L in TEST_LENGTHS:
        val = lens.get(L)
        if base and val:
            delta = val - base
            row  += f"  {'+' if delta >= 0 else ''}{delta:<7.1f}"
        else:
            row  += f"  {'N/A':<8s}"
    print(row)
print('─'*len(header))

# ── 3. Line plot ─────────────────────────────────────────────────────────────
COLORS  = {'Sinusoidal': '#e74c3c', 'RoPE': '#f39c12', 'ALiBi': '#27ae60', 'Relative': '#2980b9'}
MARKERS = {'Sinusoidal': 'X', 'RoPE': 's', 'ALiBi': 'o', 'Relative': '^'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#c9d1d9', labelsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')

# Left: absolute PPL
ax = axes[0]
for pe, lens in results.items():
    xs = [L for L in TEST_LENGTHS if lens.get(L) is not None]
    ys = [lens[L] for L in xs]
    ax.plot(xs, ys, marker=MARKERS.get(pe,'o'), color=COLORS.get(pe,'#888'),
            linewidth=2, markersize=8, label=pe)
ax.axvline(512, color='#ffffff33', linestyle='--', linewidth=1)
ax.text(520, ax.get_ylim()[0] if ax.get_ylim()[0] > 0 else 10,
        'L_train=512', color='#888888', fontsize=8)
ax.set_xlabel('L_test (tokens)', color='#c9d1d9', fontsize=11)
ax.set_ylabel('Validation Perplexity (↓)', color='#c9d1d9', fontsize=11)
ax.set_title('Absolute Perplexity vs Context Length', color='#e6edf3', fontsize=13)
ax.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=9)
ax.set_xticks(TEST_LENGTHS)

# Right: normalized (PPL / PPL@512)
ax = axes[1]
for pe, lens in results.items():
    base = lens.get(512)
    if not base:
        continue
    xs = [L for L in TEST_LENGTHS if lens.get(L) is not None]
    ys = [lens[L] / base for L in xs]
    ax.plot(xs, ys, marker=MARKERS.get(pe,'o'), color=COLORS.get(pe,'#888'),
            linewidth=2, markersize=8, label=pe)
ax.axhline(1.0, color='#ffffff55', linestyle='--', linewidth=1)
ax.axvline(512, color='#ffffff33', linestyle='--', linewidth=1)
ax.set_xlabel('L_test (tokens)', color='#c9d1d9', fontsize=11)
ax.set_ylabel('Relative PPL (PPL / PPL@512) (↓)', color='#c9d1d9', fontsize=11)
ax.set_title('Normalised Extrapolation Degradation', color='#e6edf3', fontsize=13)
ax.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=9)
ax.set_xticks(TEST_LENGTHS)

plt.suptitle('Positional Encoding Extrapolation Test  (trained at L=512)',
             color='#e6edf3', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()

os.makedirs(os.path.join(REPO, 'outputs/extrap_plots'), exist_ok=True)
plot_path = os.path.join(REPO, 'outputs/extrap_plots/extrapolation_results.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'✅ Plot saved → {plot_path}')

# ── 4. Save JSON ──────────────────────────────────────────────────────────────
json_path = os.path.join(REPO, 'outputs/extrap_plots/extrapolation_results.json')
with open(json_path, 'w') as f:
    json.dump(results, f, indent=4)
print(f'✅ Results saved → {json_path}')

### A.7 — Log Extrapolation Results to W&B

In [ ]:
try:
    import wandb

    run = wandb.init(
        project='SAiDL-Core-ML',
        name='extrapolation_test_summary',
        tags=['extrapolation', 'positional_encoding'],
    )

    # Flat metrics
    log_dict = {}
    for pe, lens in results.items():
        for L, ppl in lens.items():
            if ppl:
                log_dict[f'extrap/{pe}/ppl_L{L}'] = ppl
    wandb.log(log_dict)

    # Summary table artifact
    cols = ['PE'] + [f'PPL L={L}' for L in TEST_LENGTHS] + [f'ΔPPL L={L}' for L in TEST_LENGTHS[1:]]
    rows = []
    for pe, lens in results.items():
        base = lens.get(512) or 1
        row  = [pe] + [lens.get(L, None) for L in TEST_LENGTHS]
        row += [round((lens.get(L, base) - base), 2) if lens.get(L) else None for L in TEST_LENGTHS[1:]]
        rows.append(row)
    wandb.log({'extrap/summary_table': wandb.Table(columns=cols, data=rows)})

    wandb.finish()
    print('✅ Results logged to W&B')
except Exception as e:
    print(f'⚠ W&B logging skipped: {e}')

---
## SECTION B — Convolution + Attention Hybrids (Phase 1.4)

> **Protocol (from assignment):** Combine your **best attention variant** (from Part 2)  
> with your **best positional encoding** (from Part 3) under three 1D-Conv topologies.

Set `BEST_ATTN` and `BEST_POS` below to match your Part 1 / Part 2 results.  
Defaults are `sliding_window` + `alibi` (strong general defaults from the assignment blueprint).

| Run | Hybrid Type | What changes |
|-----|------------|-------------|
| B.1 | `conv_before_attn` | DepthwiseSepConv1d prepended before every attention sub-layer |
| B.2 | `gated_conv_ffn` | Standard attn + GLU-style gated depthwise-conv FFN (replaces MLP) |
| B.3 | `interleaved` | Even layers = PureConvBlock, odd layers = attention |

All three are then benchmarked against the **pure-attention best checkpoint** for a head-to-head perplexity vs compute comparison.

In [ ]:
# ── B.0  Configure best variants from Parts 1 & 2 ────────────────────────────
# UPDATE these to whatever gave the lowest val perplexity in SAiDL_colab.ipynb
BEST_ATTN = 'sliding_window'   # <- update if different
BEST_POS  = 'alibi'            # <- update if different

print(f'Using best attention : {BEST_ATTN}')
print(f'Using best positional: {BEST_POS}')

### B.1 — Hybrid: Conv Before Attention

Architecture: `x → DepthwiseSepConv1d → [Pre-LN Attn] → [Pre-LN FFN]`  
Adds cheap n-gram local context extraction before each global attention step.

In [ ]:
run_experiment('hybrid_conv_before_attn', [
    f'attention={BEST_ATTN}',
    f'positional={BEST_POS}',
    'model=hybrid',
    'model.hybrid.type=conv_before_attn',
    'model.hybrid.conv_kernel_size=3',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=hybrid_conv_before',
    'output_dir=outputs/hybrid_conv_before',
])

### B.2 — Hybrid: Gated Conv FFN

Architecture: Standard attention + **GLU-gated depthwise-separable conv FFN** replacing the standard MLP.  
Inspired by Conformer — the gate selects which local patterns propagate forward.

In [ ]:
run_experiment('hybrid_gated_conv_ffn', [
    f'attention={BEST_ATTN}',
    f'positional={BEST_POS}',
    'model=hybrid',
    'model.hybrid.type=gated_conv_ffn',
    'model.hybrid.conv_kernel_size=3',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=hybrid_gated_conv_ffn',
    'output_dir=outputs/hybrid_gated_ffn',
])

### B.3 — Hybrid: Interleaved Conv + Attention

Architecture: Even layers = **PureConvBlock** (no attention at all), odd layers = full attention block.  
Halves attention cost; local patterns handled by conv, global by attention.

In [ ]:
run_experiment('hybrid_interleaved', [
    f'attention={BEST_ATTN}',
    f'positional={BEST_POS}',
    'model=hybrid',
    'model.hybrid.type=interleaved',
    'model.hybrid.conv_kernel_size=7',   # larger kernel for conv-only layers
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=hybrid_interleaved',
    'output_dir=outputs/hybrid_interleaved',
])

---
### B.4 — Benchmarking: Hybrids vs Pure Attention

Runs `core_ml/benchmark.py` for each configuration, measuring:
- Validation perplexity at L ∈ {512, 1024}
- Inference latency (ms/batch)
- Throughput (tokens/sec)
- Peak GPU memory (MB)

In [ ]:
def run_benchmark(name, extra_overrides=None):
    base = [
        f'attention={BEST_ATTN}',
        f'positional={BEST_POS}',
        'dataset.batch_size=4',
        'dataset.num_workers=2',
    ]
    return run_experiment(f'bench_{name}', base + (extra_overrides or []),
                          script='core_ml/benchmark.py')

# Pure attention baseline (no hybrid model flag)
run_benchmark('pure_attn')

# All three hybrid types
for htype in ['conv_before_attn', 'gated_conv_ffn', 'interleaved']:
    run_benchmark(htype, extra_overrides=[
        'model=hybrid',
        f'model.hybrid.type={htype}',
    ])

### B.5 — Hybrid Comparison Table + Plot

In [ ]:
import glob

bench_dir   = os.path.join(REPO, 'benchmark_results')
bench_files = sorted(glob.glob(os.path.join(bench_dir, '*.json')))

rows = []
for path in bench_files:
    with open(path) as f:
        d = json.load(f)
    rows.append(d)

if not rows:
    print('⚠ No benchmark JSON files found. Run B.4 cells first.')
else:
    # ── Print table ───────────────────────────────────────────────────────────
    COLS = ['attention', 'positional', 'n_params_M',
            'ppl_seq512', 'ppl_seq1024', 'latency_ms', 'tokens_per_sec', 'peak_gpu_mb']

    print('\n' + '═'*95)
    print('  HYBRID vs PURE ATTENTION — BENCHMARK SUMMARY')
    print('═'*95)
    hdr = '  ' + '  '.join(f'{c:18s}' for c in COLS)
    print(hdr)
    print('  ' + '─' * (len(hdr) - 2))
    for r in rows:
        line = '  ' + '  '.join(f"{str(r.get(c,'—')):18s}" for c in COLS)
        print(line)
    print('═'*95)

    # ── Bar chart: PPL@1024 vs Latency ────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0d1117')
    for ax in (ax1, ax2):
        ax.set_facecolor('#161b22')
        ax.tick_params(colors='#c9d1d9')
        for spine in ax.spines.values():
            spine.set_edgecolor('#30363d')

    labels  = [r.get('attention','?') + '\n' + r.get('positional','?') for r in rows]
    ppls    = [r.get('ppl_seq1024') or 0 for r in rows]
    latency = [r.get('latency_ms') or 0 for r in rows]
    palette = ['#2ecc71' if 'hybrid' in str(r.get('attention',''))+str(r.get('positional',''))
               else '#3498db' for r in rows]
    # Simple colour by order
    pal2 = ['#e74c3c', '#f39c12', '#2ecc71', '#9b59b6']

    x = range(len(rows))

    bars = ax1.bar(x, ppls, color=[pal2[i % 4] for i in x], edgecolor='#0d1117', linewidth=0.5)
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(labels, fontsize=8, color='#c9d1d9')
    ax1.set_ylabel('Val Perplexity @ L=1024 (↓)', color='#c9d1d9', fontsize=11)
    ax1.set_title('Perplexity Comparison', color='#e6edf3', fontsize=13)
    for bar, val in zip(bars, ppls):
        if val:
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val:.1f}', ha='center', va='bottom', fontsize=8, color='#e6edf3')

    bars2 = ax2.bar(x, latency, color=[pal2[i % 4] for i in x], edgecolor='#0d1117', linewidth=0.5)
    ax2.set_xticks(list(x))
    ax2.set_xticklabels(labels, fontsize=8, color='#c9d1d9')
    ax2.set_ylabel('Inference Latency ms/batch (↓)', color='#c9d1d9', fontsize=11)
    ax2.set_title('Latency Comparison', color='#e6edf3', fontsize=13)
    for bar, val in zip(bars2, latency):
        if val:
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val:.1f}ms', ha='center', va='bottom', fontsize=8, color='#e6edf3')

    plt.suptitle(f'Hybrid Architectures vs Pure Attention  ({BEST_ATTN} + {BEST_POS})',
                 color='#e6edf3', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()

    os.makedirs(os.path.join(REPO, 'outputs/hybrid_plots'), exist_ok=True)
    hybrid_plot = os.path.join(REPO, 'outputs/hybrid_plots/hybrid_comparison.png')
    plt.savefig(hybrid_plot, dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print(f'✅ Hybrid plot saved → {hybrid_plot}')

### B.6 — Log Hybrid Results to W&B

In [ ]:
try:
    import wandb

    run = wandb.init(
        project='SAiDL-Core-ML',
        name='hybrid_benchmark_summary',
        tags=['hybrid', 'benchmark', 'convolution'],
    )

    for r in rows:
        prefix = f"hybrid/{r.get('attention','?')}_{r.get('positional','?')}"
        log_dict = {
            f'{prefix}/ppl_512':      r.get('ppl_seq512'),
            f'{prefix}/ppl_1024':     r.get('ppl_seq1024'),
            f'{prefix}/latency_ms':   r.get('latency_ms'),
            f'{prefix}/tok_per_sec':  r.get('tokens_per_sec'),
            f'{prefix}/peak_gpu_mb':  r.get('peak_gpu_mb'),
            f'{prefix}/n_params_M':   r.get('n_params_M'),
        }
        wandb.log({k: v for k, v in log_dict.items() if v is not None})

    if rows:
        cols_wb = list(rows[0].keys())
        data_wb = [[r.get(c) for c in cols_wb] for r in rows]
        wandb.log({'hybrid/summary_table': wandb.Table(columns=cols_wb, data=data_wb)})

    wandb.finish()
    print('✅ Hybrid results logged to W&B')
except Exception as e:
    print(f'⚠ W&B logging skipped: {e}')

---
## Summary — What to Report

### Section A  (Positional Extrapolation)

| Metric | Where to find it |
|--------|------------------|
| PPL at each L_test per PE | Cell A.6 table |
| ΔPPL degradation | Cell A.6 table |
| Extrapolation plot | `outputs/extrap_plots/extrapolation_results.png` |
| Raw JSON | `outputs/extrap_plots/extrapolation_results.json` |

**Expected finding:** ALiBi will show the flattest degradation curve (smallest ΔPPL as L_test grows). Sinusoidal will spike catastrophically at L > 512 because its embedding table has no entries for positions it never trained on.

### Section B  (Convolution Hybrids)

| Metric | Where to find it |
|--------|------------------|
| PPL@512, PPL@1024 per hybrid | Cell B.5 table |
| Latency, throughput, peak GPU | Cell B.5 table |
| Comparison bar charts | `outputs/hybrid_plots/hybrid_comparison.png` |
| Raw JSONs | `benchmark_results/*.json` |

**Expected finding:** `conv_before_attn` and `gated_conv_ffn` should match or slightly beat pure attention on perplexity while lowering latency. `interleaved` halves attention FLOPs but may incur a small PPL penalty on longer sequences.

In [ ]:
# ── Final: download all outputs as a zip ─────────────────────────────────────
import os

REPO = '/content/SAiDL-Summer-Assignment-2026'
os.system(
    f'cd {REPO} && '
    f'zip -r /content/extrap_hybrid_outputs.zip '
    f'outputs/extrap_plots outputs/hybrid_plots benchmark_results '
    f'2>/dev/null || true'
)

from google.colab import files
files.download('/content/extrap_hybrid_outputs.zip')
print('✅ Download started — extrap_hybrid_outputs.zip')